# Applied AI in Chemical and Process Engineering
## Week 4: Unsupervised Learning · Batch Crystallization & Pattern Discovery

---

### Learning Objectives
1. **Unsupervised Formulation**: Frame industrial chemical process challenges where operational logs lack quality failure labels or regime tags.
2. **Feature Scaling**: Apply `StandardScaler` to handle multi-unit process variables ($^\circ\text{C/min}$, $\text{RPM}$, $\text{g}$, $\text{hr}$) without scale distortion.
3. **Dimensionality Reduction (PCA)**: Compress high-dimensional process recipes into orthogonal principal components and physically interpret component loadings.
4. **Cluster Discovery ($k$-Means & Hierarchical)**: Determine optimal cluster counts via Elbow (Inertia) and Silhouette metrics, and visualize batch genealogy via dendrograms.
5. **Golden Batch Discovery & QC Mapping**: Connect discovered operational clusters to downstream crystal quality ($d_{50}$ particle size, % fines, yield) to establish optimal operating envelopes and diagnose root-cause failure modes.


## Part 1 · Chemical Process Problem Formulation & Crystallization Fundamentals

### 1.1 Industrial Context & Phase Equilibrium Fundamentals
In pharmaceutical and fine chemical manufacturing, **batch cooling crystallization** of an Active Pharmaceutical Ingredient (API) precipitates solid solute particles by cooling a concentrated solution under controlled agitation.

The downstream processability (filtration rate, cake washing efficiency, drying, and tableting) is directly governed by the **Crystal Size Distribution (CSD)**, which is dictated by where the batch operates on the **thermodynamic phase diagram** ($C$ vs. $T$):

<div align="center">
  <img src="./assets/crystallization.jpg" alt="Crystallization Phase Diagram & Metastable Zone" width="460">
  <p><em>Figure 1: Crystallization Phase Diagram showing the Stable, Metastable, and Labile zones separated by the Solubility and Spinodal curves.</em></p>
</div>

#### Phase Boundaries & Operational Outcomes:
* **Solubility Curve (Black Line · Thermodynamic Equilibrium $C^*(T)$)**: Separates the **Stable Zone (Undersaturated)** ($C < C^*$, where crystals dissolve) from the supersaturated region.
* **Metastable Zone (MSZW · Target "Golden Batch")** ($C^* \le C < C_{\text{spinodal}}$): The solution is supersaturated (driving force $\Delta C = C - C^* > 0$), but spontaneous primary nucleation is kinetically hindered ($B \approx 0$). Charging **seed crystals** provides initial surface area ($A_s$) for controlled **crystal growth** ($G = k_g \Delta C^g$) $\rightarrow$ produces large, uniform crystals ($d_{50} > 350\,\mu\text{m}$), minimal fines ($< 6\%$), and high yield ($> 92\%$) for rapid filtration.
* **Spinodal Curve (Red Line · $C_{\text{spinodal}}(T)$) & Labile Zone** ($C \ge C_{\text{spinodal}}$): Exceeding the metastable limit collapses the nucleation barrier, triggering catastrophic **primary nucleation** ($B = k_b \sigma^b, b \approx 3\text{--}5$) $\rightarrow$ **Failure Mode: Crash Cooling**, generating billions of sub-micron fines ($> 15\%$) that blind filter cloths.

---

### 1.2 Recipe Variables & Physical Mechanisms in Batch Operations

In our industrial crystallizer, 6 operating recipe parameters determine the phase trajectory and shear environment:

| Recipe Feature | Physical Role & Impact on Phase Trajectory |
|---|---|
| `Cooling_Rate_C_min` ($^\circ\text{C/min}$) | Governs supersaturation generation rate. Rapid cooling outpaces crystal growth, pushing the batch across the red Spinodal Curve into the **Labile Zone** (*Crash Cooling*). |
| `Agitation_RPM` ($\text{RPM}$) | Ensures suspension uniformity and heat transfer. Excessive impeller shear causes crystal collisions and attrition $\rightarrow$ secondary breakage into fines (*Shear Attrition*). |
| `Seed_Mass_g` ($\text{g}$) | Provides initial crystal surface area ($A_s$) to consume supersaturation through growth and suppress nucleation inside the Metastable Zone. |
| `Seed_Temp_C` ($^\circ\text{C}$) | Seeding temperature. Must be within the Metastable Zone (too hot $\rightarrow$ seeds dissolve in the Stable Zone; too cold $\rightarrow$ primary nucleation occurs prior to seeding). |
| `Initial_Conc_g_L` ($\text{g/L}$) | Starting solute concentration. Dictates total crystallizable mass and initial distance to the solubility boundary. |
| `Batch_Time_hr` ($\text{hr}$) | Residence time for desupersaturation exhaustion to reach equilibrium solubility $C^*(T_{\text{final}})$ and maximize yield. |

---

### 1.3 The Unsupervised Learning Challenge in Chemical Operations

> [!IMPORTANT]
> **The Industrial Reality: Historian Logs Lack Operational Regime Labels**:
> In chemical plants, historians continuously record recipes and sensor telemetry across 120 production batches, but **no labels tell engineers which batch belongs to what operational regime**.
> 
> * **Dimensionality Reduction (PCA)**: Compress the 6 correlated recipe dimensions into orthogonal physical axes (Kinetic Cooling Path vs. Hydrodynamic Shear).
> * **Clustering ($k$-Means)**: Group unlabeled batches into distinct operational regimes, discover the **Golden Batch envelope**, and isolate root causes of batch failures.


### Vibe-Coding Prompt 1 · Environment Setup & Data Ingestion

**Prompt to LLM:**

```text
Role: Process Data Scientist in Fine Chemicals / Pharma
Context: Ingesting unlabeled batch cooling crystallization historian records from 'https://raw.githubusercontent.com/dissabnd/Applied-AI-CPE-UG-2026/refs/heads/main/data/Batch_Crystallization_Dataset.csv'.
Task:
1. Import numpy, pandas, matplotlib.pyplot, and seaborn.
2. Load 'Batch_Crystallization_Dataset.csv' into a DataFrame `df`.
3. Display the first 10 rows and print the dataset shape and summary statistics.
```


## Part 2 · Feature Standardization (Scaling)

> [!IMPORTANT]
> **Why Standardization is Mandatory for Distance-Based ML**:
> Algorithms like PCA and $k$-Means rely on Euclidean distances:
> $$d(x_i, x_j) = \sqrt{\sum_{k=1}^p (x_{ik} - x_{jk})^2}$$
> If unscaled, `Agitation_RPM` (values ranging from $150$ to $600$) will dominate the distance calculations by a factor of $10,000\times$ compared to `Cooling_Rate_C_min` (values from $0.2$ to $1.8$).
> Standardizing transforms each variable to zero mean and unit variance ($z = \frac{x - \mu}{\sigma}$).


### Vibe-Coding Prompt 2 · Process Feature Standardization

**Prompt to LLM:**

```text
Role: Applied AI Engineer
Context: Standardizing 6 crystallization recipe features:
         ['Cooling_Rate_C_min', 'Agitation_RPM', 'Seed_Mass_g', 'Seed_Temp_C', 'Initial_Conc_g_L', 'Batch_Time_hr']
Task:
1. Extract the 6 recipe features into matrix `X`.
2. Apply `StandardScaler` from scikit-learn to produce `X_scaled`.
3. Verify that all features in `X_scaled` have mean ≈ 0 and std ≈ 1.
```


## Part 3 · Dimensionality Reduction via Principal Component Analysis (PCA)

Principal Component Analysis (PCA) projects our 6-dimensional operating space onto orthogonal axes that maximize captured variance:
* **Score Plot ($PC_1$ vs. $PC_2$)**: Shows where individual batches sit in reduced physical coordinate space.
* **Loadings Plot**: Shows which original process variables contribute to each Principal Component.

```
High-Dimensional Space (6 Variables) ──► [ Orthogonal Rotation ] ──► 2D Physical Representation
  (Cooling, RPM, Seed, Temp, Conc, Time)                              (PC1: Kinetic Mode, PC2: Hydrodynamic Mode)
```


### Vibe-Coding Prompt 3 · PCA Decomposition, Scree Plot & Loadings Biplot

**Prompt to LLM:**

```text
Role: Chemometrics & Process Modeling Specialist
Context: Scaled crystallization dataset `X_scaled` with 6 features.
Task:
1. Fit `PCA(n_components=6)` and print the explained variance ratio of each component.
2. Plot a 2-panel figure:
   - Panel A: Scree plot (cumulative explained variance vs. number of components).
   - Panel B: PCA Loadings Heatmap for PC1 and PC2 showing feature contributions.
3. Fit `PCA(n_components=2)` and compute 2D scores (`X_pca`).
```


### Chemical Engineer's Physical Interpretation of PCA Loadings:
1. **$PC_1$ (Kinetic Cooling Driving Force)**: Heavy positive loading on `Cooling_Rate_C_min` ($+0.51$) and strong negative loadings on `Batch_Time_hr` ($-0.46$), `Seed_Mass_g` ($-0.45$), and `Seed_Temp_C` ($-0.47$).
   * High $PC_1$ $\rightarrow$ Aggressive, fast cooling with insufficient seed crystal mass.
   * Low $PC_1$ $\rightarrow$ Gentle cooling, longer residence time, and well-seeded crystal growth.
2. **$PC_2$ (Hydrodynamic Agitation & Supersaturation)**: Heavy positive loading on `Agitation_RPM` ($+0.68$) and `Initial_Conc_g_L` ($+0.64$).
   * High $PC_2$ $\rightarrow$ High shear mixing intensity and high initial supersaturation.


## Part 4 · Unsupervised Regime Discovery ($k$-Means Clustering)

We now apply $k$-Means clustering to automatically group similar batches in the scaled operating space.
To determine the true number of operational regimes without guessing, we calculate:
1. **Elbow Method (Inertia / Within-Cluster Sum of Squares)**: Measures cluster compactness.
2. **Silhouette Score**: Measures how well-separated and distinct clusters are (ranges from $-1$ to $+1$, where higher is better).


### Vibe-Coding Prompt 4 · Elbow & Silhouette Optimization + $k$-Means Fitting

**Prompt to LLM:**

```text
Role: Machine Learning Engineer
Context: Scaled crystallization dataset `X_scaled`.
Task:
1. Iterate $k$ from 2 to 6, recording Inertia and Silhouette Score for each $k$.
2. Plot a 2-panel diagnostic chart (Inertia Elbow Plot & Silhouette Score Plot).
3. Fit `KMeans(n_clusters=4, random_state=42, n_init=10)` on `X_scaled`.
4. Store the discovered cluster IDs in `df['Cluster']`.
5. Plot the 2D PCA Score Plot colored by the discovered clusters.
```


## Part 5 · Chemical Engineering Profiling & "Golden Batch" Discovery

> [!NOTE]
> **Connecting Unsupervised Clusters to Downstream Crystal Quality**:
> Remember: We did **NOT** use `Mean_Crystal_Size_um`, `Fines_Pct`, or `Yield_Pct` to train PCA or $k$-Means.
> Now we evaluate how the unsupervised clusters map to actual physical quality outcomes to discover the **Golden Batch envelope** and diagnose failure modes.


### Vibe-Coding Prompt 5 · Cluster Profiling & Quality Validation

**Prompt to LLM:**

```text
Role: Lead Crystallization Process Engineer
Context: DataFrame `df` containing recipe variables, cluster assignments, and QC outputs.
Task:
1. Group `df` by `Cluster` and compute the mean of operating variables and product quality metrics.
2. Display the cluster summary table.
3. Generate a 3-panel boxplot comparing the clusters across:
   - `Mean_Crystal_Size_um` (Target: > 350 μm)
   - `Fines_Pct` (Target: < 6%)
   - `Yield_Pct` (Target: > 92%)
4. Deduce the physical mechanism behind each cluster and assign meaningful engineering labels.
```


## Part 6 · Summary of Discovered Operational Regimes

Based on our unsupervised clustering analysis, we have successfully mapped the 4 operational modes of the crystallization unit:

| Cluster | Discovered Regime Name | Recipe Characteristics | Physical Mechanism | Quality Outcome |
|---|---|---|---|---|
| **Cluster 1** | **Golden Batch** | Moderate cooling ($0.47^\circ\text{C/min}$), gentle agitation ($238\,\text{RPM}$), high seed ($10.0\,\text{g}$) | Controlled seeded growth within Metastable Zone | Large crystals ($d_{50} \approx 385\,\mu\text{m}$), minimal fines ($4.1\%$), high yield ($93.9\%$) |
| **Cluster 3** | **Shear Attrition** | High agitation ($531\,\text{RPM}$), standard cooling & seed | Excessive impeller shear breaks crystal needles | Moderate size ($d_{50} \approx 215\,\mu\text{m}$), elevated fines ($21.0\%$) |
| **Cluster 0** | **Crash Cooling** | Fast cooling ($1.45^\circ\text{C/min}$), low seed ($3.5\,\text{g}$), cold seeding | Uncontrolled primary nucleation into labile zone | Fine dust ($d_{50} \approx 108\,\mu\text{m}$), severe fines ($40.1\%$), filter blinding |
| **Cluster 2** | **Incomplete Growth** | Very slow cooling ($0.21^\circ\text{C/min}$), low conc ($128\,\text{g/L}$), short time | Insufficient driving force & residence time | Sub-optimal yield ($77.8\%$), unrecovered solute in mother liquor |

---

### Key Takeaways for Applied AI in Chemical Engineering
1. **Unsupervised learning discovered the true operational regimes without needing manual labels.**
2. **PCA transformed 6 correlated sensor/recipe variables into 2 actionable physical axes** (Kinetic cooling path vs. Hydrodynamic shear).
3. **Clustering empowers root-cause analysis:** When a plant experiences filtration blinding, engineers can immediately check if the batch landed in the *Crash Cooling* or *Shear Attrition* cluster.

